In [ ]:
# Pareto solution count analysis

from __future__ import annotations

import ast
import logging
from pathlib import Path

import pandas as pd

logger = logging.getLogger(__name__)
if not logging.getLogger().hasHandlers():
    logging.basicConfig(level=logging.INFO, format="%(message)s")


class ParetoSolutionAnalyzer:
    def __init__(
        self,
        run_ids=(1000, 5000, 10000, 50000, 100000),
        input_dir="output/TBPO",
        output_dir="output/Solution_Counts",
    ):
        self.run_ids = list(run_ids)
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.pareto_data: dict[int, dict[str, pd.DataFrame]] = {}

        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _resolve_paths(self, run_id: int) -> tuple[Path, Path | None]:
        legacy_summary = self.input_dir / f"{run_id}_pareto_optimal_solutions_summary.csv"
        legacy_full_csv = self.input_dir / f"{run_id}_pareto_optimal_solutions.csv"
        if legacy_summary.exists():
            return legacy_summary, (legacy_full_csv if legacy_full_csv.exists() else None)

        run_dir = self.input_dir / str(run_id)
        summary = run_dir / "pareto_optimal_solutions_summary.csv"
        full_parquet = run_dir / "pareto_optimal_solutions.parquet"
        full_csv = run_dir / "pareto_optimal_solutions.csv"
        if full_parquet.exists():
            full = full_parquet
        elif full_csv.exists():
            full = full_csv
        else:
            full = None
        return summary, full

    def load_pareto_frontiers(self) -> bool:
        loaded = 0
        for run_id in self.run_ids:
            summary_path, full_path = self._resolve_paths(int(run_id))
            if not summary_path.exists():
                continue

            try:
                summary_df = pd.read_csv(summary_path)
                entry: dict[str, pd.DataFrame] = {"summary": summary_df}
                if full_path is not None:
                    if full_path.suffix.lower() == ".parquet":
                        entry["full"] = pd.read_parquet(full_path, engine="pyarrow")
                    else:
                        entry["full"] = pd.read_csv(full_path)
                self.pareto_data[int(run_id)] = entry
                loaded += 1
            except Exception as exc:
                logger.warning(f"Skip run_id={run_id}: {exc}")

        if loaded == 0:
            logger.error("No Pareto data loaded. Check input_dir and run_ids.")
            return False
        logger.info(f"Loaded Pareto data: {loaded}/{len(self.run_ids)}")
        return True

    def analyze_solution_counts(self) -> pd.DataFrame | None:
        if not self.pareto_data:
            logger.error("No Pareto data loaded. Call load_pareto_frontiers() first.")
            return None

        rows: list[dict[str, int]] = []
        for run_id, data in self.pareto_data.items():
            if "full" in data and "Number_of_Sensors" in data["full"].columns:
                counts = (
                    data["full"]
                    .groupby("Number_of_Sensors", dropna=False)
                    .size()
                    .reset_index(name="Solution_Count")
                )
                for _, r in counts.iterrows():
                    rows.append(
                        {
                            "Run_ID": int(run_id),
                            "Sensor_Count": int(r["Number_of_Sensors"]),
                            "Solution_Count": int(r["Solution_Count"]),
                        }
                    )
                continue

            summary_df = data["summary"]
            if "Number_of_Sensors" not in summary_df.columns:
                continue

            if "Num_Equivalent_Solutions" in summary_df.columns:
                for _, r in summary_df.iterrows():
                    rows.append(
                        {
                            "Run_ID": int(run_id),
                            "Sensor_Count": int(r["Number_of_Sensors"]),
                            "Solution_Count": int(r["Num_Equivalent_Solutions"]),
                        }
                    )
                continue

            if "Equivalent_Solutions" in summary_df.columns:
                for _, r in summary_df.iterrows():
                    value = r["Equivalent_Solutions"]
                    count = 1
                    if isinstance(value, str):
                        try:
                            parsed = ast.literal_eval(value)
                            if isinstance(parsed, list):
                                count = len(parsed)
                        except (ValueError, SyntaxError):
                            count = 1
                    rows.append(
                        {
                            "Run_ID": int(run_id),
                            "Sensor_Count": int(r["Number_of_Sensors"]),
                            "Solution_Count": int(count),
                        }
                    )
                continue

            counts = (
                summary_df.groupby("Number_of_Sensors", dropna=False)
                .size()
                .reset_index(name="Solution_Count")
            )
            for _, r in counts.iterrows():
                rows.append(
                    {
                        "Run_ID": int(run_id),
                        "Sensor_Count": int(r["Number_of_Sensors"]),
                        "Solution_Count": int(r["Solution_Count"]),
                    }
                )

        if not rows:
            logger.error("No solution counts produced. Check input files/columns.")
            return None

        solution_counts = pd.DataFrame(rows).sort_values(["Sensor_Count", "Run_ID"])
        pivot = solution_counts.pivot_table(
            index="Sensor_Count", columns="Run_ID", values="Solution_Count", aggfunc="mean"
        ).sort_index()

        solution_counts.to_csv(self.output_dir / "pareto_solution_counts.csv", index=False)
        pivot.to_csv(self.output_dir / "pareto_solution_counts_pivot.csv")
        logger.info(f"Wrote: {self.output_dir}")

        self.solution_counts = solution_counts
        self.solution_pivot = pivot
        return solution_counts

    def run(self) -> dict[str, pd.DataFrame | None]:
        if not self.load_pareto_frontiers():
            return {"solution_counts": None, "solution_pivot": None}
        return {
            "solution_counts": self.analyze_solution_counts(),
            "solution_pivot": getattr(self, "solution_pivot", None),
        }

analyzer = ParetoSolutionAnalyzer(run_ids=[1000, 5000, 10000, 50000, 100000], input_dir="output/TBPO")
analysis_results = analyzer.run()

In [ ]:
# Jaccard Analysis

from __future__ import annotations

import ast
import logging
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)
if not logging.getLogger().hasHandlers():
    logging.basicConfig(level=logging.INFO, format="%(message)s")

output_dir = Path("output/Jaccard_Analysis")
output_dir.mkdir(parents=True, exist_ok=True)


def parse_sensor_ids(value) -> set[str]:
    if pd.isna(value):
        return set()
    if not isinstance(value, str):
        return set()
    s = value.strip()
    if not s:
        return set()
    try:
        parsed = ast.literal_eval(s)
    except (ValueError, SyntaxError):
        parsed = None
    if isinstance(parsed, list):
        return {str(item).strip() for item in parsed if str(item).strip()}
    if isinstance(parsed, tuple):
        return {str(item).strip() for item in parsed if str(item).strip()}
    return set()


def resolve_summary_path(base_dir: Path, run_id: int) -> Path | None:
    p = base_dir / str(run_id) / "pareto_optimal_solutions_summary.csv"
    if p.exists():
        return p

    p = base_dir / f"{run_id}_pareto_optimal_solutions_summary.csv"
    if p.exists():
        return p
    return None


def calculate_inter_run_jaccard_stats(run_ids, base_dir: str | Path) -> pd.DataFrame:
    base_dir = Path(base_dir)
    solutions: dict[int, dict[int, list[set[str]]]] = {}
    valid_run_ids: list[int] = []

    for run_id in run_ids:
        run_id = int(run_id)
        summary_path = resolve_summary_path(base_dir, run_id)
        if summary_path is None:
            continue
        df = pd.read_csv(summary_path)
        sensor_col = next((c for c in ("Selected_Sensor_IDs", "Sensor_IDs") if c in df.columns), None)
        if sensor_col is None or "Number_of_Sensors" not in df.columns:
            continue
        run_solutions: dict[int, list[set[str]]] = {}
        for _, row in df.iterrows():
            k = int(row["Number_of_Sensors"])
            run_solutions.setdefault(k, []).append(parse_sensor_ids(row[sensor_col]))
        solutions[run_id] = run_solutions
        valid_run_ids.append(run_id)

    if len(valid_run_ids) < 2:
        return pd.DataFrame()

    stats_rows: list[dict[str, object]] = []
    for m_i, m_j in combinations(sorted(valid_run_ids), 2):
        common_k = set(solutions[m_i].keys()) & set(solutions[m_j].keys())
        for k in sorted(common_k):
            sols_i = solutions[m_i][k]
            sols_j = solutions[m_j][k]
            if not sols_i or not sols_j:
                continue
            sims = []
            for si in sols_i:
                for sj in sols_j:
                    union = si.union(sj)
                    sims.append(len(si.intersection(sj)) / len(union) if union else 1.0)
            stats_rows.append(
                {
                    "M_i": m_i,
                    "M_j": m_j,
                    "Sensor_Count": k,
                    "Avg_Similarity": float(np.mean(sims)) if sims else np.nan,
                    "Max_Similarity": float(np.max(sims)) if sims else np.nan,
                    "Min_Similarity": float(np.min(sims)) if sims else np.nan,
                }
            )

    return pd.DataFrame(stats_rows)


run_ids = [1000, 5000, 10000, 50000, 100000]

tbpo_stats = calculate_inter_run_jaccard_stats(run_ids, base_dir="output/TBPO")
cspo_stats = calculate_inter_run_jaccard_stats(run_ids, base_dir="output/CSPO")

tbpo_path = output_dir / "tbpo_jaccard_stats.csv"
cspo_path = output_dir / "cspo_jaccard_stats.csv"

tbpo_stats.to_csv(tbpo_path, index=False)
cspo_stats.to_csv(cspo_path, index=False)

logger.info(f"Wrote: {tbpo_path}")
logger.info(f"Wrote: {cspo_path}")